# URLSentra — Member 4: Model Evaluation + SVM
**Owner:** Dharmasiri S.O.R. · **Module:** EC5203 Machine Learning · **Proposal:** sections 4.9, 4.10 and 4.12

**Status (updated 2026-11-24 for the progress evaluation):** the comparison now uses the group's agreed
protocol: the group-aware split (`data.split_index`, near-duplicate URLs never cross train/test), grouped
5-fold training CV (`evaluate.cv_folds(X, y, groups)`), and the agreed feature sets `url_base` (URL-only)
and `url_engineered` (URL-only + engineered URL features, the **primary** set). `URLSimilarityIndex` and
`IsHTTPS` are in neither set.

> **SVM = PROVISIONAL.** `svm.BEST_C = 0.1` was chosen by the contributor's earlier development run
> (ungrouped `StratifiedKFold`, 2,000 training rows, the old "original" features). It has **not** been
> retuned under the grouped protocol, so SVM numbers are reported separately and are not part of the
> official three-model comparison (Logistic Regression, Decision Tree, Random Forest; see
> `reports/official_comparison.csv`, written by `python main.py`).

**Protocol:** target `1 = phishing`; model selection uses training rows only; test data are never
evaluated during an ordinary **Run All**.

## 1. Settings and imports
Place `PhiUSIIL_Phishing_URL_Dataset.csv` under `data/raw/` (download: https://archive.ics.uci.edu/dataset/967). Run the notebook from the repository's `notebooks/` directory or the project root. `SAMPLE` limits a *development* run.

**Sample CV numbers in this notebook are diagnostics, not final results.** The official numbers come from
`python main.py` (full data) and are saved in `reports/`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise RuntimeError("Open this notebook from URLSentra's root or notebooks/ folder")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown
from sklearn.base import clone

from src import config, data, features, evaluate
from src.models import logreg, tree, forest, svm

SAMPLE = 3000  # entire cleaned data: None; smaller quick development run: 1500
SVM_TUNE_TRAIN_ROWS = 2000
RUN_FINAL_TEST = False  # IMPORTANT: keep False for ordinary Run All
FINAL_PROTOCOL_APPROVED = False  # final test requires the group's explicit approval
PRIMARY = features.PRIMARY_FEATURE_SET  # "url_engineered" (group decision 2026-11-24)
BASE = "url_base"                      # URL-only control set

print("Project root:", ROOT)
print("SAMPLE:", SAMPLE, "• RUN_FINAL_TEST:", RUN_FINAL_TEST)

## 2. Load/clean data and prepare the common grouped training split
Preprocessing stays in Member 1's `src/data.py`. `data.split_index()` is the group-aware split shared by
every member (whole URL groups go to train or test). The two feature sets are built by
`features.build_feature_sets()` on **training rows only** and use exactly the same rows; the URL group
ids of those rows are passed to every CV call so no group is split between fitting and validation folds.

In [ ]:
if not config.RAW_FILE.is_file():
    raise FileNotFoundError(
        f"Dataset not found: {config.RAW_FILE}. Download it from the UCI link above "
        "and place the CSV under data/raw/. Do not commit the dataset."
    )
df = data.load_clean_data()
if SAMPLE is not None and SAMPLE < len(df):
    df = df.sample(n=SAMPLE, random_state=config.RANDOM_STATE).reset_index(drop=True)
df = features.add_features(df)

train_idx, _test_idx = data.split_index(df)
del _test_idx  # DO NOT evaluate the test set during normal Run All
train_groups = data.group_ids(df).loc[train_idx]
y_train = data.get_target(df).loc[train_idx]
sets = features.build_feature_sets(df.loc[train_idx])     # training rows only
assert "URLSimilarityIndex" not in sets[PRIMARY] and "IsHTTPS" not in sets[PRIMARY]

feature_sets = {
    BASE: (df.loc[train_idx, sets[BASE]], y_train),
    PRIMARY: (df.loc[train_idx, sets[PRIMARY]], y_train),
}
display(pd.DataFrame({
    "feature_set": list(feature_sets),
    "n_train_rows": [len(x) for x, _ in feature_sets.values()],
    "n_features": [x.shape[1] for x, _ in feature_sets.values()],
    "n_url_groups": train_groups.nunique(),
}))
print("Engineered columns added by url_engineered:", [c for c in sets[PRIMARY] if c not in sets[BASE]])
print("Phishing positive-class ratio in training data:", round(float(y_train.mean()), 4))

## 3. SVM methodology and hyperparameter tuning (training only, PROVISIONAL)
We use a **linear support vector classifier** (`LinearSVC`) because a kernel SVM may be expensive at ~235k rows. The `StandardScaler` is **inside** the calibrated classifier so scaling and probability calibration are refitted inside each cross-validation fold. The calibrated estimator supports `predict_proba()` needed for ROC-AUC and phishing risk scores.

This cell re-runs the C search over `[0.01, 0.1, 1, 10]` on a group-preserving subset of the training rows
(`data.group_subset`) with **grouped** folds (`svm.tune(..., groups=...)`). It is a development check only:
`svm.BEST_C` is **not** changed from the contributor's provisional 0.1, and the SVM remains outside the official
comparison until it is retuned on an approved, larger training sample.

In [ ]:
tune_idx = data.group_subset(train_groups, SVM_TUNE_TRAIN_ROWS)
X_tune, y_tune = feature_sets[PRIMARY][0].loc[tune_idx], y_train.loc[tune_idx]
print(f"Tuning SVM on {len(X_tune):,} TRAINING rows ({train_groups.loc[tune_idx].nunique():,} URL groups, no test rows) ...")
search = svm.tune(X_tune, y_tune, n_jobs=1, groups=train_groups.loc[tune_idx])
cv_tuning = svm.tuning_table(search)
display(cv_tuning.round(5))
best_c = float(search.best_params_["estimator__linearsvc__C"])
print("Best C on this grouped TRAINING CV subset:", best_c, "| svm.BEST_C (provisional, used by main.py):", svm.BEST_C)
print("Mean CV F1 for selected C:", round(float(search.best_score_), 5))

## 4. Shared model comparison: url_base vs url_engineered (grouped TRAINING CV)
All four models are compared with **the same grouped five-fold training cross-validation**, on the same rows. We use the frozen estimators from each member's module (`build()`); the SVM uses the provisional `svm.BEST_C`. No test-set scores are used here. On a small `SAMPLE` the numbers are noisy; the full-data CV for the three main models is in `reports/cv_results.csv`.

In [ ]:
models = {
    "Logistic Regression": logreg.build(),
    "Decision Tree": tree.build(),
    "Random Forest": forest.build(),
    "SVM (provisional)": svm.build(),
}
cv_results = evaluate.compare_training_models(feature_sets, models, groups=train_groups)
columns = ["feature_set", "model"] + [f"{m}_mean" for m in evaluate.CV_METRICS]
display(cv_results[columns].round(5))
# Keep full std columns for reproducibility and confidence/variability reporting.
display(cv_results[["feature_set", "model", "f1_mean", "f1_std", "roc_auc_mean", "roc_auc_std"]].round(5))

## 5. Model comparison chart and url_base-vs-url_engineered deltas
The plotted bars show **grouped CV F1 means**, not held-out final scores. Absolute differences can be misleading near 1.0; always inspect class balance and the format reference rule (`reports/format_baseline.csv`), not just overall accuracy.

In [ ]:
deltas = evaluate.feature_delta_table(cv_results, base=BASE, other=PRIMARY)
display(deltas[["model", f"f1_mean_{BASE}", f"f1_mean_{PRIMARY}", "delta_f1_mean"]].round(6))
fig1 = evaluate.plot_feature_comparison(
    cv_results, metric="f1", filename=f"member4_train_cv_feature_comparison_sample_{SAMPLE}.png" if SAMPLE
    else "member4_train_cv_feature_comparison.png", base=BASE, other=PRIMARY,
)
plt.show()

## 6. ROC and confusion matrices — INTERNAL validation only
To visualize models without looking at the true test data, the first **grouped** CV fold of the training rows is used: its fitting part trains each model and its validation part is scored (no URL group is in both). Every ROC curve and confusion matrix uses this identical validation partition; it is **diagnostic**, not an official final estimate.

In [ ]:
X_primary = feature_sets[PRIMARY][0]
fit_pos, valid_pos = evaluate.cv_folds(X_primary, y_train, train_groups)[0]
X_inner, X_valid = X_primary.iloc[fit_pos], X_primary.iloc[valid_pos]
y_inner, y_valid = y_train.iloc[fit_pos], y_train.iloc[valid_pos]
assert not set(train_groups.iloc[fit_pos]) & set(train_groups.iloc[valid_pos])
fitted_inner = {}
for name, prototype in models.items():
    print("Inner validation fit:", name)
    fitted_inner[name] = clone(prototype).fit(X_inner, y_inner)

suffix = f"_sample_{SAMPLE}" if SAMPLE else ""
fig2 = evaluate.plot_roc_curves(
    fitted_inner, X_valid, y_valid,
    title="URLSentra: training-only grouped inner-validation ROC (url_engineered)",
    filename=f"member4_train_validation_roc{suffix}.png",
)
plt.show()
fig3 = evaluate.plot_confusion_matrices(
    fitted_inner, X_valid, y_valid,
    title="Training-only grouped inner-validation confusion matrices (url_engineered)",
    filename=f"member4_train_validation_confusion{suffix}.png",
)
plt.show()

## 7. Results and interpretation (automatically derived from the actual run)
This section displays the observed numbers from **the actual data loaded above**. It does not invent numerical results. Record the SVM's best C and the difference between original and engineered features after executing this notebook.

**Interpretation rules:** Positive label is phishing. Recall tells us how many phishing URLs are detected; precision tells us what fraction of phishing alerts are correct; F1 balances them; ROC-AUC measures ranking across decision thresholds. Compare the uncertainty/variation across folds. A training sample is not a substitute for an independent external-domain validation.

In [ ]:
primary_cv = cv_results.loc[cv_results.feature_set == PRIMARY].copy()
best_row = primary_cv.loc[primary_cv.f1_mean.idxmax()]
svm_primary = primary_cv.loc[primary_cv.model == "SVM (provisional)"].iloc[0]
svm_deltas = deltas.loc[deltas.model == "SVM (provisional)"].iloc[0]

print("OBSERVED GROUPED TRAINING-CV SUMMARY (sample; not final test results)")
print(f"Best {PRIMARY} mean CV F1: {best_row['model']} = {best_row['f1_mean']:.5f}")
print(f"SVM (provisional) {PRIMARY} CV F1: {svm_primary.f1_mean:.5f} ± {svm_primary.f1_std:.5f}")
print(f"SVM {PRIMARY}-minus-{BASE} CV F1: {svm_deltas['delta_f1_mean']:+.5f}")
if (cv_results.f1_mean >= 0.99).any():
    display(Markdown(
        "**WARNING — near-perfect F1 observed.** Compare with the format reference rule "
        "(`reports/format_baseline.csv`): every legitimate PhiUSIIL URL is a bare https://www homepage, "
        "so URL-only scores near 0.99 mostly reflect how the dataset was collected, not general "
        "phishing-detection ability."
    ))
display(Markdown(
    "**Limitation:** these are sampled grouped training-CV results. The SVM's C is provisional "
    "(earlier ungrouped 2,000-row tuning). No held-out test metric is calculated in this notebook run."
))

## 8. Optional FINAL held-out test (disabled by default)
The official held-out results of all models (SVM row labelled `provisional`) are produced once by `python main.py`
and saved to `reports/results.csv`. This cell only displays them. Setting `RUN_FINAL_TEST = True` additionally refits
the provisional SVM here; do not choose C or features from that result.

In [ ]:
results_csv = config.RESULTS_DIR / "results.csv"
if results_csv.exists():
    res = pd.read_csv(results_csv)
    display(res.loc[res["model"] == "SVM", ["feature_set", "model", "protocol", "accuracy", "precision",
                                             "recall", "f1", "roc_auc", "tn", "fp", "fn", "tp"]].round(5))
else:
    print("reports/results.csv not found: run `python main.py` on the full data first.")
if RUN_FINAL_TEST:
    if SAMPLE is not None:
        raise RuntimeError("Final evaluation requires SAMPLE = None (the full cleaned dataset).")
    if not FINAL_PROTOCOL_APPROVED:
        raise RuntimeError("Final evaluation blocked: set FINAL_PROTOCOL_APPROVED = True after group approval.")
    X_train_final, X_test_final, y_train_final, y_test_final = data.split(df, sets[PRIMARY])
    final_svm = svm.build().fit(X_train_final, y_train_final)
    display(pd.DataFrame([{"model": "SVM (provisional)", "feature_set": PRIMARY,
                           **evaluate.evaluate(final_svm, X_test_final, y_test_final)}]))
else:
    print("Notebook-side held-out test SKIPPED (RUN_FINAL_TEST = False).")

## 9. Experiment log and handover notes
| Step | Why | Evidence on execution | Status |
| --- | --- | --- | --- |
| Exclude `URLSimilarityIndex` (and `IsHTTPS` from the URL-only sets) | Group decisions 2026-11-19 / 2026-11-24: label proxy and collection bias | `features.build_feature_sets()` | Applied |
| LinearSVC + calibration | Fast linear baseline and probability estimates | `svm.configured()` implementation | Implemented |
| Tune C | Choose model complexity using training CV | Section 3 (grouped CV on a training subset) | **Provisional**: `BEST_C = 0.1` from the contributor's ungrouped development run (`SAMPLE=3000`, 2,000 rows, old features); not retuned under the grouped protocol |
| Compare four models on `url_base` vs `url_engineered` | Fair common grouped CV | Sections 4–5 | Updated 2026-11-24 to the grouped protocol (sample run) |
| Validation ROC and confusion matrices | Visualize false positives and missed phishing | Section 6 (first grouped fold) | Training-only diagnostics |
| Official full-data and test reporting | Unbiased reported final result | `python main.py` → `reports/results.csv` (SVM row `protocol = provisional`) | Done by main.py; SVM not in the official comparison |
| Full-data run 2026-11-24 (provisional SVM, C = 0.1) | Record only | `reports/results.csv` | Test F1 0.9849 (`url_base`), 0.9838 (`url_engineered`), 0.9992 (`extended`); below LR/DT/RF and the format rule on the URL-only sets |

**Remaining for the SVM:** retune C with `svm.tune(X, y, groups=...)` on an approved, larger group-preserving training sample, update `BEST_C` only with the verified optimum, then rerun `python main.py`.

**Known limitations:** the dataset's legitimate URLs are all bare https://www homepages (collection format, see `reports/format_baseline.csv` and `reports/format_controlled.csv`), sample-based tuning, and the SVM's provisional C.

**Suggested verification commands (repo root):**
```bash
.venv\Scripts\activate
jupyter notebook notebooks/03_evaluation_svm.ipynb
python main.py --sample 20000 --explain-rows 0
```
Do **not** commit CSV datasets, fitted models or notebook outputs.